<a href="https://colab.research.google.com/github/G0rav/machine_learning_explained_visually_free/blob/main/06-class-imbalance/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class Imbalance: 98% Accuracy, 0% Fraud Caught

**[Watch the video](https://youtu.be/ZVEzPWNNPeg)** · a standalone special in *Machine
Learning Explained Visually*.

Everything from the video, runnable. Three layers:

1. **Follow along** — every number in the video, in code, in the same order.
2. **Experiment** — change the fraud prevalence and the false-negative cost,
   the two knobs the video itself asks you to turn.
3. **Challenge** — write the threshold selector and the SMOTE interpolation
   step yourself before checking them.

Nothing here imports from anywhere. Only `numpy` is required. Run it top to
bottom.

Every section asserts the number the video put on screen. If a check ever
fails, the video and this notebook have drifted apart and one of them is
wrong.

---

# Layer 1 — Follow along

In [1]:
import numpy as np

PASSED = 0
def check(label, got, want, tol=1e-9):
    # Assert a value the video states, and say so.
    global PASSED
    got_f, want_f = float(got), float(want)
    assert abs(got_f - want_f) <= tol, f"{label}: {got_f!r} != {want_f!r}"
    PASSED += 1
    print(f"ok   {label:52s} {got_f:.9g}")

def check_tuple(label, got, want):
    global PASSED
    got_t, want_t = tuple(got), tuple(want)
    assert got_t == want_t, f"{label}: {got_t!r} != {want_t!r}"
    PASSED += 1
    print(f"ok   {label:52s} {got_t}")

## Part 1 — Imbalanced targets and the majority baseline

Five thousand transactions, exactly one hundred of them fraud. A classifier
that predicts *legitimate* for every single row, without looking at either
input feature, is the majority baseline: whatever the majority class already
is, guess only that.

The dataset is synthetic and deterministic — two continuous features,
`amount_deviation` and `transaction_velocity`, drawn from two overlapping
normal distributions — so it regenerates identically here and in the video.

In [2]:
SEED = 240922
N_LEGIT, N_FRAUD = 4_900, 100
N_TOTAL = N_LEGIT + N_FRAUD

def make_dataset(n_legit=N_LEGIT, n_fraud=N_FRAUD, seed=SEED):
    rng = np.random.default_rng(seed)
    legit = rng.multivariate_normal(
        mean=[3.3, 3.0], cov=[[2.2, 0.45], [0.45, 2.0]], size=n_legit)
    fraud = rng.multivariate_normal(
        mean=[5.7, 5.4], cov=[[3.0, 0.7], [0.7, 2.6]], size=n_fraud)
    x = np.clip(np.vstack([legit, fraud]), 0.0, 10.0)
    y = np.concatenate([np.zeros(n_legit, dtype=int), np.ones(n_fraud, dtype=int)])
    order = rng.permutation(len(y))
    return x[order], y[order]

def stratified_indices(y, seed=SEED):
    # Exact 60/20/20 train/validation/test split, per class.
    rng = np.random.default_rng(seed)
    partitions = [[], [], []]
    for label in (0, 1):
        idx = np.flatnonzero(np.asarray(y) == label)
        rng.shuffle(idx)
        train_end = int(0.6 * len(idx))
        validation_end = int(0.8 * len(idx))
        pieces = (idx[:train_end], idx[train_end:validation_end], idx[validation_end:])
        for target, piece in zip(partitions, pieces):
            target.extend(piece.tolist())
    for target in partitions:
        rng.shuffle(target)
    return tuple(np.asarray(t, dtype=int) for t in partitions)

X, Y = make_dataset()
check("transactions", len(Y), N_TOTAL)
check("fraud rows", int(Y.sum()), N_FRAUD)

TRAIN_IDX, VAL_IDX, TEST_IDX = stratified_indices(Y)
TRAIN_X, TRAIN_Y = X[TRAIN_IDX], Y[TRAIN_IDX]
VAL_X, VAL_Y = X[VAL_IDX], Y[VAL_IDX]
TEST_X, TEST_Y = X[TEST_IDX], Y[TEST_IDX]

check("train size", len(TRAIN_Y), 3_000)
check("validation size", len(VAL_Y), 1_000)
check("test size", len(TEST_Y), 1_000)
check("validation fraud rows", int(VAL_Y.sum()), 20)
check("test fraud rows", int(TEST_Y.sum()), 20)

all_idx = np.concatenate([TRAIN_IDX, VAL_IDX, TEST_IDX])
assert len(np.unique(all_idx)) == N_TOTAL, "partitions overlap or miss rows"
print("ok   train/validation/test partitions are disjoint and cover every row")

ok   transactions                                         5000
ok   fraud rows                                           100
ok   train size                                           3000
ok   validation size                                      1000
ok   test size                                            1000
ok   validation fraud rows                                20
ok   test fraud rows                                      20
ok   train/validation/test partitions are disjoint and cover every row


In [3]:
def confusion(y_true, y_pred):
    # Confusion counts with fraud (1) as the positive class.
    y_true, y_pred = np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int)
    return {
        "tn": int(np.sum((y_true == 0) & (y_pred == 0))),
        "fp": int(np.sum((y_true == 0) & (y_pred == 1))),
        "fn": int(np.sum((y_true == 1) & (y_pred == 0))),
        "tp": int(np.sum((y_true == 1) & (y_pred == 1))),
    }

def metrics(y_true, y_pred):
    counts = confusion(y_true, y_pred)
    total = sum(counts.values())
    pred_pos = counts["tp"] + counts["fp"]
    actual_pos = counts["tp"] + counts["fn"]
    return {
        **counts,
        "accuracy": (counts["tp"] + counts["tn"]) / total,
        "precision": counts["tp"] / pred_pos if pred_pos else 0.0,
        "recall": counts["tp"] / actual_pos if actual_pos else 0.0,
    }

majority_pred = np.zeros(len(VAL_Y), dtype=int)   # always "legitimate"
baseline = metrics(VAL_Y, majority_pred)

check_tuple("majority baseline confusion (tn, fp, fn, tp)",
            (baseline["tn"], baseline["fp"], baseline["fn"], baseline["tp"]),
            (980, 0, 20, 0))
check("majority baseline accuracy", baseline["accuracy"], 0.98)
check("majority baseline recall", baseline["recall"], 0.0)

print(f"\n{100*baseline['accuracy']:.0f}% accuracy, "
      f"{100*baseline['recall']:.0f}% fraud recall — the whole problem in two numbers.")

ok   majority baseline confusion (tn, fp, fn, tp)         (980, 0, 20, 0)
ok   majority baseline accuracy                           0.98
ok   majority baseline recall                             0

98% accuracy, 0% fraud recall — the whole problem in two numbers.


## Part 2 — Classifier scores and the decision threshold

A classifier does not have to output a class directly — it can output a
score between 0 and 1, and a **decision threshold** turns that score into a
prediction. Moving the threshold changes predictions without retraining
anything: the scores, and their ranking, stay exactly the same.

We fit an ordinary (unweighted) logistic regression on the training rows —
standardized using *training* statistics only, so no information from
validation or test leaks into preprocessing.

In [4]:
TRAIN_MEAN, TRAIN_STD = TRAIN_X.mean(axis=0), TRAIN_X.std(axis=0)

def standardize(x):
    return (np.asarray(x, dtype=float) - TRAIN_MEAN) / TRAIN_STD

TRAIN_Z = standardize(TRAIN_X)
VAL_Z = standardize(VAL_X)
TEST_Z = standardize(TEST_X)

def sigmoid(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def fit_logreg(x, y, sample_weight=None, max_iter=80, tol=1e-11):
    # Unregularized logistic regression via Newton's method with a line search.
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    design = np.column_stack([x, np.ones(len(x))])
    weights = (np.ones(len(x)) if sample_weight is None
               else np.asarray(sample_weight, dtype=float))
    theta = np.zeros(design.shape[1])

    def loss(at):
        score = design @ at
        terms = np.logaddexp(0.0, score) - y * score
        return float(np.sum(weights * terms) / np.sum(weights))

    for _ in range(max_iter):
        score = design @ theta
        p = sigmoid(score)
        gradient = design.T @ (weights * (p - y)) / weights.sum()
        curvature = weights * p * (1.0 - p)
        hessian = (design.T * curvature) @ design / weights.sum()
        step = np.linalg.solve(hessian + np.eye(hessian.shape[0]) * 1e-12, gradient)
        before = loss(theta)
        scale = 1.0
        while loss(theta - scale * step) > before and scale > 1e-8:
            scale *= 0.5
        theta -= scale * step
        if np.max(np.abs(scale * step)) < tol:
            break
    return theta

def scores(theta, x):
    design = np.column_stack([np.asarray(x, dtype=float), np.ones(len(x))])
    return sigmoid(design @ np.asarray(theta, dtype=float))

THETA_PLAIN = fit_logreg(TRAIN_Z, TRAIN_Y)
VAL_SCORES_PLAIN = scores(THETA_PLAIN, VAL_Z)
print("fitted the plain classifier and scored every validation row")

fitted the plain classifier and scored every validation row


In [5]:
# The fraud validation row whose score lands nearest 0.35 — the same row the
# video reads its worked score from.
fraud_val_idx = np.flatnonzero(VAL_Y == 1)
example_idx = fraud_val_idx[np.argmin(np.abs(VAL_SCORES_PLAIN[fraud_val_idx] - 0.35))]
example_score = float(VAL_SCORES_PLAIN[example_idx])
print(f"example fraud row score: {example_score:.3f}")

for threshold in (0.500, 0.300):
    prediction = "FRAUD" if example_score >= threshold else "LEGITIMATE"
    print(f"threshold {threshold:.3f}  ->  {prediction}")

assert (example_score >= 0.500) != (example_score >= 0.300) or example_score < 0.300 \
    or example_score >= 0.500, "the threshold move should be able to flip this row"
print("\nNotice: the score never moved. Only the threshold did.")

example fraud row score: 0.353
threshold 0.500  ->  LEGITIMATE
threshold 0.300  ->  FRAUD

Notice: the score never moved. Only the threshold did.


## Part 3 — Class weighting

Class weighting is a training-time intervention: instead of every row
contributing equally to the loss, each row is weighted by the inverse of its
class frequency, so the rare class counts for more during fitting. Nothing
about the training *rows* changes — only how much each one counts.

In [6]:
BALANCED_WEIGHTS = {
    0: len(TRAIN_Y) / (2.0 * np.sum(TRAIN_Y == 0)),
    1: len(TRAIN_Y) / (2.0 * np.sum(TRAIN_Y == 1)),
}
check("balanced legitimate weight", BALANCED_WEIGHTS[0], 3_000 / (2 * 2_940), tol=1e-12)
check("balanced fraud weight", BALANCED_WEIGHTS[1], 3_000 / (2 * 60), tol=1e-12)

sample_weight = np.where(TRAIN_Y == 1, BALANCED_WEIGHTS[1], BALANCED_WEIGHTS[0])
THETA_WEIGHTED = fit_logreg(TRAIN_Z, TRAIN_Y, sample_weight=sample_weight)
VAL_SCORES_WEIGHTED = scores(THETA_WEIGHTED, VAL_Z)
print(f"fraud weight is {BALANCED_WEIGHTS[1]:.1f}x the legitimate weight")

ok   balanced legitimate weight                           0.510204082
ok   balanced fraud weight                                25
fraud weight is 25.0x the legitimate weight


## Part 4 — Random resampling and SMOTE

Three training-*sample* interventions, each applied to the training partition
only — validation and test rows are never touched:

- **Random oversampling** duplicates minority rows until the classes are equal.
- **Random undersampling** removes majority rows until the classes are equal.
- **SMOTE** creates *synthetic* minority rows by interpolating between a real
  minority row and one of its nearest minority neighbors.

In [7]:
def random_oversample(x, y, seed=240924):
    x, y = np.asarray(x, float), np.asarray(y, int)
    rng = np.random.default_rng(seed)
    minority, majority = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    additions = rng.choice(minority, len(majority) - len(minority), replace=True)
    indices = np.concatenate([np.arange(len(y)), additions])
    rng.shuffle(indices)
    return x[indices], y[indices], additions

def random_undersample(x, y, seed=240926):
    x, y = np.asarray(x, float), np.asarray(y, int)
    rng = np.random.default_rng(seed)
    minority = np.flatnonzero(y == 1)
    kept_majority = rng.choice(np.flatnonzero(y == 0), len(minority), replace=False)
    indices = np.concatenate([kept_majority, minority])
    rng.shuffle(indices)
    return x[indices], y[indices], kept_majority

def smote(x, y, seed=240927, neighbors=5):
    x, y = np.asarray(x, float), np.asarray(y, int)
    rng = np.random.default_rng(seed)
    minority = x[y == 1]
    majority_count = int(np.sum(y == 0))
    distances = np.linalg.norm(minority[:, None, :] - minority[None, :, :], axis=2)
    neighbor_indices = np.argsort(distances, axis=1)[:, 1:neighbors + 1]
    needed = majority_count - len(minority)
    synthetic = np.empty((needed, x.shape[1]), dtype=float)
    for row in range(needed):
        source_index = int(rng.integers(len(minority)))
        neighbor_index = int(rng.choice(neighbor_indices[source_index]))
        lam = float(rng.random())
        synthetic[row] = minority[source_index] + lam * (minority[neighbor_index] - minority[source_index])
    return np.vstack([x, synthetic]), np.concatenate([y, np.ones(needed, dtype=int)])

OVER_X, OVER_Y, OVER_ADDITIONS = random_oversample(TRAIN_Z, TRAIN_Y)
UNDER_X, UNDER_Y, UNDER_KEPT = random_undersample(TRAIN_Z, TRAIN_Y)
SMOTE_X, SMOTE_Y = smote(TRAIN_Z, TRAIN_Y)

check_tuple("oversampled class counts",
            (int(np.sum(OVER_Y == 0)), int(np.sum(OVER_Y == 1))), (2_940, 2_940))
check("oversampling duplicate rows added", len(OVER_ADDITIONS), 2_880)
check_tuple("undersampled class counts",
            (int(np.sum(UNDER_Y == 0)), int(np.sum(UNDER_Y == 1))), (60, 60))
check("undersampling majority rows removed", 2_940 - len(UNDER_KEPT), 2_880)
check_tuple("SMOTE class counts",
            (int(np.sum(SMOTE_Y == 0)), int(np.sum(SMOTE_Y == 1))), (2_940, 2_940))

THETA_OVER = fit_logreg(OVER_X, OVER_Y)
THETA_UNDER = fit_logreg(UNDER_X, UNDER_Y)
THETA_SMOTE = fit_logreg(SMOTE_X, SMOTE_Y)
VAL_SCORES_OVER = scores(THETA_OVER, VAL_Z)
VAL_SCORES_UNDER = scores(THETA_UNDER, VAL_Z)
VAL_SCORES_SMOTE = scores(THETA_SMOTE, VAL_Z)
print("all three resampled candidates fitted and scored on the untouched validation rows")

ok   oversampled class counts                             (2940, 2940)
ok   oversampling duplicate rows added                    2880
ok   undersampled class counts                            (60, 60)
ok   undersampling majority rows removed                  2880
ok   SMOTE class counts                                   (2940, 2940)
all three resampled candidates fitted and scored on the untouched validation rows


## Part 5 — Precision, recall, threshold selection & the verdict

Missing a fraud transaction costs **100** units here; flagging a legitimate
one costs **5**. That asymmetry is what a decision threshold should be chosen
against — not accuracy, and not a default of 0.5.

In [8]:
FN_COST, FP_COST = 100, 5

def operational_cost(counts, fn_cost=FN_COST, fp_cost=FP_COST):
    return fn_cost * counts["fn"] + fp_cost * counts["fp"]

def threshold_rows(y_true, s, fn_cost=FN_COST, fp_cost=FP_COST):
    s = np.asarray(s, dtype=float)
    thresholds = np.unique(np.concatenate([[1.0], s, [0.0]]))[::-1]
    rows = []
    for t in thresholds:
        pred = (s >= t).astype(int)
        row = metrics(y_true, pred)
        row["threshold"] = float(t)
        row["cost"] = int(operational_cost(row, fn_cost, fp_cost))
        rows.append(row)
    return rows

def select_threshold(y_true, s, fn_cost=FN_COST, fp_cost=FP_COST):
    rows = threshold_rows(y_true, s, fn_cost, fp_cost)
    return min(rows, key=lambda r: (r["cost"], -r["threshold"]))

def average_precision(y_true, s):
    y_true, s = np.asarray(y_true, dtype=int), np.asarray(s, dtype=float)
    order = np.argsort(-s, kind="mergesort")
    ranked = y_true[order]
    positives = np.cumsum(ranked == 1)
    ranks = np.arange(1, len(ranked) + 1)
    hits = positives[ranked == 1] / ranks[ranked == 1]
    return float(hits.mean()) if len(hits) else 0.0

at_half = metrics(VAL_Y, (VAL_SCORES_PLAIN >= 0.5).astype(int))
at_half["cost"] = int(operational_cost(at_half))
check_tuple("plain classifier at threshold 0.5 (tn, fp, fn, tp)",
            (at_half["tn"], at_half["fp"], at_half["fn"], at_half["tp"]), (979, 1, 17, 3))
check("plain classifier cost at threshold 0.5", at_half["cost"], 1_705)

selected_plain = select_threshold(VAL_Y, VAL_SCORES_PLAIN)
check("plain classifier's selected threshold", selected_plain["threshold"],
      0.08411466692185766, tol=1e-9)
print(f"sweeping the threshold drops validation cost from 1,705 to "
      f"{selected_plain['cost']} at t = {selected_plain['threshold']:.3f}")
check("plain classifier recall at selected threshold", selected_plain["recall"], 0.90)
check("plain classifier precision at selected threshold", round(selected_plain["precision"], 2), 0.29)

ap_plain = average_precision(VAL_Y, VAL_SCORES_PLAIN)
check("plain classifier average precision", ap_plain, 0.4678919737301272, tol=1e-9)

ok   plain classifier at threshold 0.5 (tn, fp, fn, tp)   (979, 1, 17, 3)
ok   plain classifier cost at threshold 0.5               1705
ok   plain classifier's selected threshold                0.0841146669
sweeping the threshold drops validation cost from 1,705 to 420 at t = 0.084
ok   plain classifier recall at selected threshold        0.9
ok   plain classifier precision at selected threshold     0.29
ok   plain classifier average precision                   0.467891974


In [9]:
CANDIDATES = {
    "plain": VAL_SCORES_PLAIN,
    "class weighted": VAL_SCORES_WEIGHTED,
    "random oversampling": VAL_SCORES_OVER,
    "random undersampling": VAL_SCORES_UNDER,
    "SMOTE": VAL_SCORES_SMOTE,
}

EXPECTED_VALIDATION = {
    "plain": (936, 44, 2, 18, 420),
    "class weighted": (933, 47, 2, 18, 435),
    "random oversampling": (932, 48, 2, 18, 440),
    "random undersampling": (933, 47, 2, 18, 435),
    "SMOTE": (932, 48, 2, 18, 440),
}

print(f"{'candidate':22s} {'tn':>5} {'fp':>5} {'fn':>5} {'tp':>5} {'cost':>6}   ap")
selected_by_name = {}
for name, s in CANDIDATES.items():
    selected = select_threshold(VAL_Y, s)
    selected_by_name[name] = selected
    actual = (selected["tn"], selected["fp"], selected["fn"], selected["tp"], selected["cost"])
    check_tuple(f"{name} selected validation result", actual, EXPECTED_VALIDATION[name])
    ap = average_precision(VAL_Y, s)
    print(f"{name:22s} {selected['tn']:5d} {selected['fp']:5d} {selected['fn']:5d} "
          f"{selected['tp']:5d} {selected['cost']:6d}   {ap:.3f}")

winner = min(selected_by_name, key=lambda n: (selected_by_name[n]["cost"],
                                               -average_precision(VAL_Y, CANDIDATES[n])))
check("the winning candidate is the unchanged classifier, threshold-tuned",
      1.0 if winner == "plain" else 0.0, 1.0)
print(f"\nWinner: {winner!r} — all five candidates detect 18/20 validation fraud rows;"
      f"\nweighting and resampling only add more false positives here.")

candidate                 tn    fp    fn    tp   cost   ap
ok   plain selected validation result                     (936, 44, 2, 18, 420)
plain                    936    44     2    18    420   0.468


ok   class weighted selected validation result            (933, 47, 2, 18, 435)
class weighted           933    47     2    18    435   0.446


ok   random oversampling selected validation result       (932, 48, 2, 18, 440)
random oversampling      932    48     2    18    440   0.451


ok   random undersampling selected validation result      (933, 47, 2, 18, 435)
random undersampling     933    47     2    18    435   0.436
ok   SMOTE selected validation result                     (932, 48, 2, 18, 440)
SMOTE                    932    48     2    18    440   0.448
ok   the winning candidate is the unchanged classifier, threshold-tuned 1

Winner: 'plain' — all five candidates detect 18/20 validation fraud rows;
weighting and resampling only add more false positives here.


With the classifier and threshold fixed on validation evidence alone, the
held-out test set gets evaluated exactly once.

In [10]:
TEST_SCORES_PLAIN = scores(THETA_PLAIN, TEST_Z)
final_threshold = selected_by_name["plain"]["threshold"]
test_pred = (TEST_SCORES_PLAIN >= final_threshold).astype(int)
test_result = metrics(TEST_Y, test_pred)
test_result["cost"] = int(operational_cost(test_result))

check_tuple("final test confusion (tn, fp, fn, tp)",
            (test_result["tn"], test_result["fp"], test_result["fn"], test_result["tp"]),
            (947, 33, 7, 13))
check("final test cost", test_result["cost"], 865)

baseline_test = metrics(TEST_Y, np.zeros(len(TEST_Y), dtype=int))
baseline_test_cost = int(operational_cost(baseline_test))
check("majority-baseline test cost", baseline_test_cost, 2_000)

print(f"majority baseline:  recall {100*baseline_test['recall']:.0f}%   cost {baseline_test_cost}")
print(f"selected classifier: recall {100*test_result['recall']:.0f}%   cost {test_result['cost']}")

ok   final test confusion (tn, fp, fn, tp)                (947, 33, 7, 13)
ok   final test cost                                      865
ok   majority-baseline test cost                          2000
majority baseline:  recall 0%   cost 2000
selected classifier: recall 65%   cost 865


## Part 6 — Leakage-safe model selection

Two ways this comparison could quietly become unfair, both worth seeing in
code rather than only in words.

In [11]:
# --- wrong order: resample before splitting -------------------------------
def stratified_indices_fraction(y, seed=SEED, fraction=0.8):
    rng = np.random.default_rng(seed)
    train_idx, holdout_idx = [], []
    for label in (0, 1):
        idx = np.flatnonzero(np.asarray(y) == label)
        rng.shuffle(idx)
        cut = int(fraction * len(idx))
        train_idx.extend(idx[:cut].tolist())
        holdout_idx.extend(idx[cut:].tolist())
    return np.asarray(train_idx), np.asarray(holdout_idx)

# Oversample the WHOLE dataset first, then split it.
leaky_x, leaky_y, leaky_additions = random_oversample(X, Y, seed=1)
leaky_train_idx, leaky_holdout_idx = stratified_indices_fraction(leaky_y)

# A duplicated fraud row's original and its copy can now land on opposite
# sides of the split — count copies whose original landed on the opposite
# side from the copy itself, by identity.
original_len = len(Y)
train_set, holdout_set = set(leaky_train_idx.tolist()), set(leaky_holdout_idx.tolist())
leak_count = 0
for i, src in enumerate(leaky_additions):
    copy_idx = original_len + i
    src_in_train = int(src) in train_set
    copy_in_train = copy_idx in train_set
    if src_in_train != copy_in_train:
        leak_count += 1

print(f"resample-before-split: {leak_count} duplicated fraud rows have their "
      f"original and copy on opposite sides of the split")
assert leak_count > 0, "resampling before the split should produce some cross-split duplicates"

# --- correct order: split, THEN resample only the training fold ----------
# TRAIN_Z / TRAIN_Y above were already split before OVER_X / OVER_Y were
# built from them, so no synthetic or duplicated row can reach VAL_X or
# TEST_X — resampling only ever reads the training partition.
assert set(TRAIN_IDX.tolist()).isdisjoint(set(VAL_IDX.tolist()) | set(TEST_IDX.tolist()))
print("split-then-resample (what Parts 3 and 4 did): validation and test rows"
      " were never touched by any sampler")

resample-before-split: 1476 duplicated fraud rows have their original and copy on opposite sides of the split
split-then-resample (what Parts 3 and 4 did): validation and test rows were never touched by any sampler


In [12]:
# The same separation applies to threshold selection itself: the threshold
# must be chosen from validation scores, never from test outcomes.
threshold_from_validation = select_threshold(VAL_Y, VAL_SCORES_PLAIN)["threshold"]
assert threshold_from_validation == final_threshold
print(f"threshold {final_threshold:.3f} was selected from validation data only, "
      f"then applied to test data exactly once — the order Part 6 argues for")

# And every candidate above got its OWN threshold selected the same way,
# rather than comparing four candidates at a shared default of 0.5.
for name in CANDIDATES:
    assert selected_by_name[name]["threshold"] != 0.5 or name == "__never__"
print("every candidate was compared at its own selected threshold, not a shared 0.5")

threshold 0.084 was selected from validation data only, then applied to test data exactly once — the order Part 6 argues for
every candidate was compared at its own selected threshold, not a shared 0.5


## Part 7 — Intervention sequence (recap)

Diagnose, then compare, then select the smallest intervention that clears
the bar, then test once:

In [13]:
print("diagnose  : 98% accuracy, 0% fraud recall — accuracy alone hid the failure")
print("compare   : threshold tuning, class weighting, oversampling, undersampling, SMOTE")
print(f"select    : {winner!r}, validation cost {selected_by_name[winner]['cost']} "
      f"(vs 1,705 at the default threshold, vs {baseline_test_cost} for the baseline)")
print(f"test once : recall 0% -> {100*test_result['recall']:.0f}%, "
      f"cost {baseline_test_cost} -> {test_result['cost']}")

print(f"\nLayer 1 complete — {PASSED} checks passed, all of them numbers the video states.")

diagnose  : 98% accuracy, 0% fraud recall — accuracy alone hid the failure
compare   : threshold tuning, class weighting, oversampling, undersampling, SMOTE
select    : 'plain', validation cost 420 (vs 1,705 at the default threshold, vs 2000 for the baseline)
test once : recall 0% -> 65%, cost 2000 -> 865

Layer 1 complete — 32 checks passed, all of them numbers the video states.


---

# Layer 2 — Experiment

The video's own closing challenge: change the fraud prevalence and the
false-negative cost, then run the same comparison again.

In [14]:
def majority_baseline_accuracy(prevalence):
    # With no model at all, accuracy from always predicting the majority
    # class is exactly 1 minus the minority prevalence.
    return 1.0 - prevalence

print(f"{'fraud prevalence':>18}   majority-baseline accuracy")
for prevalence in (0.01, 0.02, 0.05, 0.10, 0.20):
    print(f"{100*prevalence:>16.0f}%   {100*majority_baseline_accuracy(prevalence):>24.1f}%")

check("accuracy at this dataset's own 2% prevalence",
      majority_baseline_accuracy(N_FRAUD / N_TOTAL), baseline["accuracy"])
print("\nThe more imbalanced the classes, the more misleading accuracy alone becomes.")

  fraud prevalence   majority-baseline accuracy
               1%                       99.0%
               2%                       98.0%
               5%                       95.0%
              10%                       90.0%
              20%                       80.0%
ok   accuracy at this dataset's own 2% prevalence         0.98

The more imbalanced the classes, the more misleading accuracy alone becomes.


Now the other knob: how does the selected threshold — and the winning
candidate — move if a missed fraud case costs less, or more, than 100 units?

In [15]:
def compare_all(fn_cost, fp_cost=FP_COST):
    # Re-run the Part 5 comparison at a different false-negative cost.
    results = {}
    for name, s in CANDIDATES.items():
        results[name] = select_threshold(VAL_Y, s, fn_cost=fn_cost, fp_cost=fp_cost)
    winner_name = min(results, key=lambda n: results[n]["cost"])
    return results, winner_name

print(f"{'FN cost':>8}   {'best threshold':>15}   {'winner':>22}   {'cost':>6}")
for fn_cost in (20, 50, 100, 200, 500):
    results, winner_name = compare_all(fn_cost)
    best = results[winner_name]
    print(f"{fn_cost:>8}   {best['threshold']:>15.3f}   {winner_name:>22}   {best['cost']:>6}")

 FN cost    best threshold                   winner     cost


      20             0.825           class weighted      230


      50             0.084                    plain      320


     100             0.084                    plain      420


     200             0.084                    plain      620


     500             0.621      random oversampling      995


Try your own prevalence and cost below — nothing past this cell depends on
the values from Part 1 through 7, so changing `NEW_FN_COST` and re-running
this cell is safe.

In [16]:
NEW_FN_COST = 30       # try 10, 500, or anything else
NEW_FP_COST = 5

results, winner_name = compare_all(NEW_FN_COST, NEW_FP_COST)
for name, r in results.items():
    marker = "  <- winner" if name == winner_name else ""
    print(f"{name:22s} threshold {r['threshold']:.3f}   cost {r['cost']:5d}{marker}")

plain                  threshold 0.107   cost   275
class weighted         threshold 0.825   cost   270  <- winner
random oversampling    threshold 0.824   cost   280
random undersampling   threshold 0.859   cost   280
SMOTE                  threshold 0.834   cost   270


---

# Layer 3 — Challenge

**1. Write your own threshold selector.** Given validation labels and scores,
find the threshold that minimizes `100 * FN + 5 * FP` by sweeping every
distinct score, without looking at `select_threshold` above.

In [17]:
def my_select_threshold(y_true, s, fn_cost=100, fp_cost=5):
    # your code here
    raise NotImplementedError

# Uncomment once you have an implementation:
# mine = my_select_threshold(VAL_Y, VAL_SCORES_PLAIN)
# check("my_select_threshold matches the reference", mine["threshold"],
#       selected_by_name["plain"]["threshold"], tol=1e-9)

**2. Explain a decision.** Using `compare_all` above, find a false-negative
cost where the winner is **not** `"plain"`. Write one sentence saying which
validation number changed your decision.

In [18]:
# your_fn_cost = ...
# results, winner_name = compare_all(your_fn_cost)
# print(winner_name, results[winner_name])

**3. Implement SMOTE's interpolation rule yourself.** A synthetic row is
`source + lambda * (neighbor - source)` for a random `lambda` in `[0, 1)`.
Reproduce one synthetic point from `smote()` above by fixing the random draw.

In [19]:
def my_smote_point(source, neighbor, lam):
    # your code here
    raise NotImplementedError

# minority = TRAIN_Z[TRAIN_Y == 1]
# mine = my_smote_point(minority[0], minority[1], 0.4)

---

That is the class-imbalance special: accuracy alone hid a classifier that
caught no fraud at all, and the fix that actually won on validation and test
data was the simplest one on the table — a threshold selected from cost, not
a fancier training-time intervention.

**[Watch more of the series](https://www.youtube.com/@school_whool)**